# NERISAL: DM-Count density model → ExecuTorch (.pte), Google Colab

DM-Count (NeurIPS 2020, MIT licence) estimates a **density map**: every person adds up to 1, so the sum is the count.
We use it only for packed crowds, where heads are too small to box.

This notebook:
1. downloads the official code and pretrained weights (ShanghaiTech B = street crowds, QNRF = very dense crowds)
2. checks the counts in PyTorch at full resolution and at our Pi sizes
3. exports to `.pte` at **256×144** and **432×240** and verifies each file
4. zips everything for the Builder to time on the Pi

Runtime: **CPU is fine** (Runtime → Change runtime type → CPU). About 15–20 minutes.

In [1]:
# 1. Install (ExecuTorch pinned to the Pi's version) and get the official DM-Count code + weights
!pip install -q executorch==1.6.0 gdown
!pip show executorch | grep Version
!git clone -q https://github.com/cvlab-stonybrook/DM-Count.git
!gdown -q --folder https://drive.google.com/drive/folders/10U7F4iW_aPICM5-qJq21SXLLkzlum9tX -O dm_weights
import glob, os
W = sorted(glob.glob("dm_weights/**/*.pth", recursive=True))
print("weights:", [os.path.basename(w) + f" ({os.path.getsize(w)/1e6:.0f} MB)" for w in W])

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Version: 1.6.0
weights: ['model_nwpu.pth (86 MB)', 'model_qnrf.pth (86 MB)', 'model_sh_A.pth (86 MB)', 'model_sh_B.pth (86 MB)']


In [7]:
# 2. Load a model (builds VGG-19 + DM-Count head, then loads the pretrained weights)
import sys, torch, numpy as np
sys.path.insert(0, "DM-Count")
from models import vgg19

def load(name):
    path = [w for w in W if name in os.path.basename(w).lower()]
    assert path, f"no weights matching '{name}' in {W}"
    m = vgg19()                                         # downloads ImageNet VGG-19 once (backbone shape)
    m.load_state_dict(torch.load(path[0], map_location="cpu"))
    return m.eval()

MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)
def prep(img_rgb, size=None):
    """img_rgb: HxWx3 uint8. size: (w, h) to resize to, or None for full resolution."""
    import cv2
    if size: img_rgb = cv2.resize(img_rgb, size, interpolation=cv2.INTER_AREA)
    x = (img_rgb.astype(np.float32) / 255 - MEAN) / STD
    return torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)

models = {}
for name in ("sh_b", "qnrf"):
    try:
        models[name] = load(name); print("loaded", name)
    except AssertionError as e:
        print(e)

loaded sh_b
loaded qnrf


In [3]:
# 3. Test images: the repo's examples + any crowd photos you upload (drag into the file panel, folder 'test_imgs')
import cv2
os.makedirs("test_imgs", exist_ok=True)
imgs = sorted(glob.glob("DM-Count/example_images/*") + glob.glob("test_imgs/*"))
print(len(imgs), "test images:", [os.path.basename(i) for i in imgs])

3 test images: ['1.png', '2.png', '3.png']


In [8]:
# 4. Count check: full resolution vs our Pi input sizes (shows how much shrinking the image costs)
SIZES = {"full": None, "432x240": (432, 240), "256x144": (256, 144)}
rows = []
for p in imgs:
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    for name, m in models.items():
        r = [os.path.basename(p)[:20], name]
        for s in SIZES.values():
            with torch.no_grad():
                mu, _ = m(prep(im, s))
            r.append(f"{mu.sum().item():.0f}")
        rows.append(r)
print(f"{'image':22s}{'model':6s}" + "".join(f"{k:>10s}" for k in SIZES))
for r in rows:
    print(f"{r[0]:22s}{r[1]:6s}" + "".join(f"{v:>10s}" for v in r[2:]))

image                 model       full   432x240   256x144
1.png                 sh_b         127       109        33
1.png                 qnrf         173       149       141
2.png                 sh_b         399       387       228
2.png                 qnrf         461       306       218
3.png                 sh_b        1112       462       191
3.png                 qnrf         789       475       243


In [9]:
# 4b. Tiling test: split the image, run each tile at 432x240, add up the counts
def tiled_count(m, im, grid, size=(432, 240)):
    H, W = im.shape[:2]
    ys = np.linspace(0, H, grid + 1).astype(int); xs = np.linspace(0, W, grid + 1).astype(int)
    total = 0.0
    for i in range(grid):
        for j in range(grid):
            with torch.no_grad():
                total += m(prep(im[ys[i]:ys[i+1], xs[j]:xs[j+1]], size))[0].sum().item()
    return total

print(f"{'image':22s}{'model':6s}{'full':>8s}{'432':>8s}{'2x2':>8s}{'3x3':>8s}")
for p in imgs:
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    for name, m in models.items():
        with torch.no_grad():
            full = m(prep(im))[0].sum().item(); one = m(prep(im, (432, 240)))[0].sum().item()
        print(f"{os.path.basename(p)[:20]:22s}{name:6s}{full:8.0f}{one:8.0f}"
              f"{tiled_count(m, im, 2):8.0f}{tiled_count(m, im, 3):8.0f}")

image                 model     full     432     2x2     3x3
1.png                 sh_b       127     109     138     153
1.png                 qnrf       173     149     163     161
2.png                 sh_b       399     387     298     318
2.png                 qnrf       461     306     417     427
3.png                 sh_b      1112     462     815    1102
3.png                 qnrf       789     475     671     784


In [13]:
# 4c. Speed vs accuracy: which tile setting keeps the count close to full size?
m = models["qnrf"]
combos = [(2, (432, 240)), (3, (432, 240)), (3, (256, 144)), (4, (256, 144))]
print(f"{'image':10s}{'full':>7s}" + "".join(f"{f'{g}x{g}@{s[0]}':>12s}" for g, s in combos))
for p in imgs:
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        full = m(prep(im))[0].sum().item()
    print(f"{os.path.basename(p)[:10]:10s}{full:7.0f}" + "".join(f"{tiled_count(m, im, g, s):12.0f}" for g, s in combos))

image        full     2x2@432     3x3@432     3x3@256     4x4@256
1.png         173         163         161         147         161
2.png         461         417         427         422         446
3.png         789         671         784         675         739


In [10]:
# 5. Export to ExecuTorch (.pte, XNNPACK) at both Pi sizes; the exported model returns the density map only
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

class DensityOnly(torch.nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        return self.m(x)[0]          # density map (1, 1, H/8, W/8); sum = count

os.makedirs("pte", exist_ok=True)
for name, m in models.items():
    for w, h in ((432, 240), (256, 144)):
        x = torch.randn(1, 3, h, w)
        with torch.no_grad():
            ep = torch.export.export(DensityOnly(m), (x,))
        prog = to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()]).to_executorch()
        out = f"pte/dmcount_{name}_{w}x{h}.pte"
        open(out, "wb").write(prog.buffer)
        print(out, f"{os.path.getsize(out)/1e6:.0f} MB")

pte/dmcount_sh_b_432x240.pte 86 MB


/content/DM-Count/models.py:24: UserWarning: `nn.functional.upsample_bilinear` is deprecated. Use `nn.functional.interpolate` instead.
  x = F.upsample_bilinear(x, scale_factor=2)


pte/dmcount_sh_b_256x144.pte 86 MB


/content/DM-Count/models.py:24: UserWarning: `nn.functional.upsample_bilinear` is deprecated. Use `nn.functional.interpolate` instead.
  x = F.upsample_bilinear(x, scale_factor=2)


pte/dmcount_qnrf_432x240.pte 86 MB


/content/DM-Count/models.py:24: UserWarning: `nn.functional.upsample_bilinear` is deprecated. Use `nn.functional.interpolate` instead.
  x = F.upsample_bilinear(x, scale_factor=2)


pte/dmcount_qnrf_256x144.pte 86 MB


In [11]:
# 6. Verify each .pte: runs, right output shape, and gives the same count as PyTorch
from executorch.runtime import Runtime
import time
test = cv2.cvtColor(cv2.imread(imgs[0]), cv2.COLOR_BGR2RGB)
for f in sorted(glob.glob("pte/*.pte")):
    name = "sh_b" if "sh_b" in f else "qnrf"; w, h = map(int, f.split("_")[-1][:-4].split("x"))
    x = prep(test, (w, h))
    meth = Runtime.get().load_program(f).load_method("forward")
    t = time.perf_counter(); d = meth.execute([x])[0]; ms = (time.perf_counter() - t) * 1000
    with torch.no_grad():
        ref = models[name](x)[0].sum().item()
    print(f"{os.path.basename(f):28s} out {tuple(d.shape)}  count .pte {d.sum().item():7.1f}  pytorch {ref:7.1f}  "
          f"({ms:.0f} ms on Colab CPU, Pi will differ)")

/content/DM-Count/models.py:24: UserWarning: `nn.functional.upsample_bilinear` is deprecated. Use `nn.functional.interpolate` instead.
  x = F.upsample_bilinear(x, scale_factor=2)


dmcount_qnrf_256x144.pte     out (1, 1, 18, 32)  count .pte   140.7  pytorch   140.7  (413 ms on Colab CPU, Pi will differ)
dmcount_qnrf_432x240.pte     out (1, 1, 30, 54)  count .pte   149.4  pytorch   149.4  (1374 ms on Colab CPU, Pi will differ)
dmcount_sh_b_256x144.pte     out (1, 1, 18, 32)  count .pte    32.6  pytorch    32.6  (311 ms on Colab CPU, Pi will differ)
dmcount_sh_b_432x240.pte     out (1, 1, 30, 54)  count .pte   108.6  pytorch   108.6  (939 ms on Colab CPU, Pi will differ)


In [12]:
# 7. Zip for the Builder (then B times them on the Pi: decision gate is < 2 s per frame)
import zipfile
with zipfile.ZipFile("nerisal_dmcount_pte.zip", "w") as z:
    for f in glob.glob("pte/*.pte"):
        z.write(f, os.path.basename(f))
print(zipfile.ZipFile("nerisal_dmcount_pte.zip").namelist())
from google.colab import files; files.download("nerisal_dmcount_pte.zip")

['dmcount_qnrf_256x144.pte', 'dmcount_qnrf_432x240.pte', 'dmcount_sh_b_256x144.pte', 'dmcount_sh_b_432x240.pte']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>